# Варіант: [Впишіть ваше місто/варіант]
Виконання завдань 1-5.

In [ ]:
import pandas as pd

# УВАГА: Підставте дані свого варіанта!
wide_df = pd.DataFrame({
    "місто": ["Київ"],
    "Січ": [75],
    "Лют": [68],
    "Бер": [55],
    "Кві": [45],
    "Тра": [30],
    "Чер": [25],
    "Лип": [20],
    "Сер": [22],
    "Вер": [35],
    "Жов": [50],
    "Лис": [65],
    "Гру": [72]
})

print("--- Wide-таблиця (вихідна) ---")
display(wide_df)

### Завдання 2. Приведення до tidy (melt)

In [ ]:
# Перетворення у tidy-формат (довгий формат)
tidy_df = wide_df.melt(id_vars="місто", var_name="місяць", value_name="хмарність")

print("--- Tidy-таблиця ---")
display(tidy_df)

print("\nФорма tidy-таблиці:", tidy_df.shape)

**Пояснення:** 
Форма таблиці стала `(12, 3)` (12 рядків). У вихідній таблиці був 1 рядок, але він містив 12 окремих спостережень (значень хмарності для різних місяців), розкиданих по стовпцях. Функція `.melt()` взяла ці 12 стовпців-місяців і "розгорнула" їх вертикально, створивши для кожного спостереження окремий рядок.

### Завдання 3. Класифікація змінних

- **Місто**: *Номінальна шкала* (Nominal). Значення є просто назвами/категоріями, між ними немає математичного відношення чи природного порядку (Київ не "більший" чи "менший" за Львів у математичному сенсі).
- **Місяць**: *Порядкова шкала* (Ordinal). Місяці є категоріями, які мають строгий логічний хронологічний порядок (Січень передує Лютому), але їх назви як текст не дозволяють виконувати арифметичні дії. (Хоча якщо рахувати час у днях, її можна вважати й інтервальною).
- **Хмарність**: *Відносна шкала* (Ratio). Вона має абсолютний нуль (0 означає повну відсутність хмарності), і ми можемо сказати, що хмарність 60 удвічі більша за хмарність 30.

### Завдання 4. Зворотне перетворення й перевірка

In [ ]:
# Повернення до wide-формату
# Зверніть увагу: pivot за замовчуванням сортує стовпці за алфавітом
back_to_wide = tidy_df.pivot(index="місто", columns="місяць", values="хмарність")

# Щоб порядок стовпців відповідав оригіналу, застосуємо reindex
original_columns = wide_df.columns[1:] # беремо назви місяців з оригіналу
back_to_wide = back_to_wide.reindex(columns=original_columns).reset_index()

# Видаляємо ім'я осі стовпців ("місяць"), яке додає pivot, для повної ідентичності
back_to_wide.columns.name = None 

print("--- Відновлена wide-таблиця ---")
display(back_to_wide)

### Завдання 5. Читання й запис (CSV та JSON)

In [ ]:
# 1. Запис у файли
tidy_df.to_csv("variant.csv", index=False)
tidy_df.to_json("variant.json", orient="records", force_ascii=False)

# 2. Читання з файлів
reloaded_csv = pd.read_csv("variant.csv")
reloaded_json = pd.read_json("variant.json")

print("--- Прочитано з CSV ---")
display(reloaded_csv.head(3))

print("\n--- Прочитано з JSON ---")
display(reloaded_json.head(3))

# 3. Демонстрація сирих (raw) даних з файлів
print("\n=== Вміст variant.csv ===")
with open("variant.csv", "r", encoding="utf-8") as f:
    print(f.read())

print("=== Вміст variant.json ===")
with open("variant.json", "r", encoding="utf-8") as f:
    print(f.read())

### Відповіді на питання:

**1. Чи можете пояснити, чому стовпець "місто" — номінальна шкала, а стовпець "хмарність" — шкала відношень?**
Стовпець "місто" містить лише назви (ярлики) для розрізнення об'єктів. Їх не можна додавати, множити чи логічно впорядковувати за зростанням (крім простого алфавітного), тому це номінальна шкала. Стовпець "хмарність" містить кількісні показники, які мають абсолютну точку відліку (справжній нуль — хмарності немає взагалі). Завдяки цьому ми можемо вираховувати відношення: наприклад, хмарність 80 балів рівно вдвічі більша, ніж 40 балів. Тому це шкала відношень.

**2. Чи розумієте, чому вихідна wide-таблиця з місяцями в стовпцях не вважається tidy, і що саме змінюється в її структурі після melt()?**
Таблиця вважається "tidy" (охайною), якщо дотримуються трьох правил: кожна змінна — це стовпець, кожне спостереження — це рядок, кожен тип спостережних одиниць — це таблиця. У wide-таблиці назви місяців були іменами стовпців, але насправді "місяць" — це змінна (ознака), а значення в клітинках — це хмарність (інша змінна). Після застосування `melt()` структура виправляється: ми перетворюємо стовпці-значення на один стовпець "місяць", а їхній вміст переносимо в новий стовпець "хмарність". Тепер кожен рядок — це одне унікальне спостереження (хмарність у конкретному місті в конкретний місяць).

**3. Чи можете описати, чим формат CSV принципово відрізняється від JSON при зберіганні однієї й тієї самої таблиці?**
- **CSV (Comma-Separated Values):** Це плоский текстовий формат. Він зберігає дані у вигляді рядків, де значення розділені комами (перший рядок — заголовки стовпців). Він дуже компактний, бо імена стовпців пишуться лише один раз.
- **JSON (JavaScript Object Notation):** Це ієрархічний структурований формат. У варіанті `orient="records"` він зберігає дані як масив словників (об'єктів). Для КОЖНОГО рядка таблиці він повторює назви стовпців у вигляді пар "ключ": "значення" (наприклад, `{"місто":"Київ","місяць":"Січ","хмарність":75}`). Через це JSON-файл займає більше місця, але він набагато краще підходить для складних вкладених структур та веброзробки.